In [10]:
import matplotlib.pyplot as plt
import seaborn as sns

from data_preprocessor import compute_metrics_baseline_feedback
from data_reader import get_all_eval_data

# Data Reading

In [11]:
df = get_all_eval_data()
metrics_df = compute_metrics_baseline_feedback(df)

/home/zaur/projects/SemSI/code/visualizations/data_preprocessor.py:84: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_copy[occurrence_cols] = df_copy[occurrence_cols].replace({'yes': 1, 'no': 0}).astype(int)
/home/zaur/projects/SemSI/code/visualizations/data_preprocessor.py:92: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  coverage = df_grouped.apply(compute_coverage_group).groupby(by=['judge', 'model', 'method']).mean() * 100


# Data Visualization

### Slope Plot

In [14]:
debug = False

plt.rcParams.update({'font.size': 12})

privacy_utility_df = metrics_df[['occurrence', 'utility']]
privacy_utility_df = privacy_utility_df.unstack('method').dropna()
privacy_utility_df.columns = ['occurrence_b', 'occurrence_f', 'utility_b', 'utility_f']
privacy_utility_df['occurrence_pct_drop'] = (privacy_utility_df['occurrence_b'] - privacy_utility_df['occurrence_f']) / privacy_utility_df['occurrence_b']
privacy_utility_df['utility_pct_drop'] = 1 - ((privacy_utility_df['utility_b'] - privacy_utility_df['utility_f']) / privacy_utility_df['utility_b'])
privacy_utility_df = privacy_utility_df.reset_index()
privacy_utility_df = privacy_utility_df[privacy_utility_df['judge'] == 'gpt-5']

# plotting
fig, ax = plt.subplots(figsize=(8, 5), dpi=100 if debug else 1200)

x_left, x_right = 0, 1  # x positions for the two columns
for _, row in privacy_utility_df.iterrows():
    xs = [x_left, x_right]
    ys = [row['occurrence_pct_drop'], row['utility_pct_drop']]
    line, = ax.plot(xs, ys, marker='o', linewidth=2, markersize=8, zorder=1000)  # use default color cycle
    color = line.get_color()
    # label letters at the endpoints, colored to match the line
    model_name = row['model'][:10] + '.' if len(row['model']) > 10 else row['model']
    if row['model'] == 'glm-4.5-air':
        ax.text(x_left + 0.03, row['occurrence_pct_drop'], model_name, va='center', ha='left', fontsize=10, fontweight='bold', color=color)
    elif row['model'] == 'grok-4.1-fast':
        ax.text(x_left - 0.03, row['occurrence_pct_drop'] + 0.02, model_name, va='center', ha='right', fontsize=10, fontweight='bold', color=color)
    else:
        ax.text(x_left - 0.03, row['occurrence_pct_drop'], model_name, va='center', ha='right', fontsize=10, fontweight='bold', color=color)
    # ax.text(x_right + 0.03, row['utility_pct_drop'], row['model'], va='center', ha='left', fontsize=6, fontweight='bold', color=color)

# vertical black separators like the example
ax.vlines(x_left, ymin=0.0, ymax=1.0, color='k', linewidth=2)
ax.vlines(x_right, ymin=0.0, ymax=1.0, color='k', linewidth=2)

# horizontal helper lines to indicate 'Low / Medium / High' bands
for y, label in zip([0.2, 0.5, 0.8], ['Low Decrease', 'Medium', 'High Decrease']):
    ax.text(-0.28, y + 0.02, label, va='center', fontsize=10)

for y, label in zip([0.2, 0.5, 0.8], ['High Decrease', 'Medium', 'Low Decrease']):
    ax.hlines(y, xmin=-0.3, xmax=1.3, linestyles='dashed', linewidth=0.8, alpha=0.6)
    ax.text(1.025, y + 0.02, label, va='center', fontsize=10)

# Title and annotations similar to the sample image
# ax.set_title('Privacy vs. Utility Trade-offs', fontsize=16, pad=12)
# ax.text(x_right + 0.05, 0.98, 'Best Both', ha='left', va='top', fontsize=10)
# ax.text(x_left - 0.05, 0.02, 'Worst Both', ha='right', va='bottom', fontsize=10)

# center annotation with arrow
ax.annotate(f'There was an average 34.6% privacy\ndecrease across the models, '
            f'while the\naverage utility decrease was 9.8%',
            xy=(0.5, 0.5), xytext=(0.4, 0.04), bbox=dict(boxstyle='square', fc="w", ec="k"))

# clean up axes
ax.set_xticks([])
ax.set_xlim(-0.3, 1.3)
ax.set_ylim(-0.03, 1.03)

# Left axis label
ax.set_ylabel('Privacy Decrease\n(Higher is better)')

# Twin right axis to show 'Utility Decrease' label (same scale)
ax_right = ax.twinx()
ax_right.set_ylim(ax.get_ylim())
ax_right.invert_yaxis()
ax_right.set_ylabel('Utility Decrease\n(Lower is better)', rotation=-90, labelpad=15, va='center')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

if not debug:
    fig.savefig('figures/privacy_utility.pdf')